# Laboratorio 1 — Exploración, preparación y regresión lineal

**Curso:** ISIS-2611 — Aprendizaje de Máquina  
**Caso:** AlpesPlanck  
**Integrantes:**  
- Nombre 1  
- Nombre 2  

---

## Objetivo

Construir y evaluar modelos de regresión lineal para predecir la **temperatura máxima diaria**, identificar las variables meteorológicas que más aportan a la predicción y analizar posibles sesgos del proceso.

> **Importante:** este notebook es una plantilla base. Cada decisión de exploración, limpieza, preparación, selección de variables y modelado debe quedar acompañada por una explicación y justificación en Markdown.


## Estructura esperada del repositorio

Se recomienda mantener el notebook dentro de la carpeta del laboratorio y conservar los datos en una subcarpeta `data/`.

```text
L1/
├── Laboratorio_1.ipynb
├── Laboratorio_1.html
├── README.md                  # opcional
├── data/
│   ├── Datos Lab 1.csv
│   ├── Datos Test Lab 1.csv
│   └── Diccionario de datos.xlsx
└── resultados/
    └── Datos Test Lab 1_predicciones.csv
```

Las rutas usadas en este notebook son **relativas**, para que funcionen al clonar el repositorio en otro computador.


# 0. Configuración e importación de librerías

En esta sección se importan las librerías necesarias para:
- manipulación de datos,
- visualización,
- preparación,
- construcción de pipelines,
- regresión lineal,
- evaluación del modelo.

Agregar otras librerías únicamente si se necesitan y explicar para qué se usan.


In [2]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Separación de datos
from sklearn.model_selection import train_test_split

# Preparación
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Modelo
from sklearn.linear_model import LinearRegression

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Utilidades
from pathlib import Path

pd.set_option("display.max_columns", None)


In [ ]:
# Rutas relativas del proyecto
DATA_DIR = Path("data")
RESULTS_DIR = Path("resultados")
RESULTS_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / "Datos Lab 1.csv"
FINAL_TEST_PATH = DATA_DIR / "Datos Test Lab 1.csv"
DICTIONARY_PATH = DATA_DIR / "Diccionario de datos.xlsx"

print("Datos de entrenamiento:", TRAIN_PATH)
print("Datos finales sin etiqueta:", FINAL_TEST_PATH)


# 1. Entendimiento del problema y de los datos

## 1.1 Contexto

**Problema:** predecir la temperatura máxima de un día a partir de variables meteorológicas.

**Tipo de aprendizaje:** supervisado.

**Tipo de tarea:** regresión, porque la variable objetivo es numérica continua.

### Preguntas iniciales

- ¿Qué representa cada registro?
- ¿Cuál es exactamente la variable objetivo?
- ¿Qué variables están disponibles antes de realizar la predicción?
- ¿Existen variables que podrían producir fuga de información (*data leakage*)?
- ¿Qué unidad tiene cada variable?


## 1.2 Diccionario de datos

Revisar primero el archivo `Diccionario de datos.xlsx`.

**Aquí deben escribir un resumen de las variables relevantes y señalar cuál será la variable objetivo.**


In [ ]:
# Cargar diccionario de datos
diccionario = pd.read_excel(DICTIONARY_PATH)
diccionario


# 2. Carga de datos


In [ ]:
# Cargar datos de entrenamiento
data = pd.read_csv(TRAIN_PATH)

print("Dimensiones:", data.shape)
data.head()


## 2.1 Observaciones iniciales

**Escribir aquí:**
- número de filas y columnas,
- qué representa una fila,
- qué tipos de variables existen,
- primeras observaciones importantes del dataset.


In [ ]:
# Información general del DataFrame
data.info()


In [ ]:
# Estadísticas descriptivas
data.describe(include="all").T


# 3. Exploración de datos (EDA)

El objetivo de esta sección es conocer el conjunto de datos antes de transformarlo.

Se debe analizar:
1. tipos de datos,
2. valores faltantes,
3. duplicados,
4. validez y posibles valores fuera de rango,
5. distribución de variables,
6. valores atípicos,
7. relaciones entre variables,
8. relación de las variables con la variable objetivo.

Después de cada análisis importante se debe escribir una conclusión.


## 3.1 Tipos de datos


In [ ]:
data.dtypes


**Análisis:**  
¿Los tipos detectados por Pandas coinciden con lo descrito en el diccionario?  
¿Hay fechas o números almacenados como texto?  


## 3.2 Completitud — valores faltantes


In [ ]:
faltantes = pd.DataFrame({
    "faltantes": data.isna().sum(),
    "porcentaje": data.isna().mean() * 100
}).sort_values("porcentaje", ascending=False)

faltantes


**Análisis y decisión preliminar:**  
Indicar qué columnas tienen valores faltantes, en qué proporción y qué estrategia podría aplicarse posteriormente.


## 3.3 Unicidad — duplicados


In [ ]:
print("Filas duplicadas:", data.duplicated().sum())

# Mostrar duplicados si existen
data[data.duplicated(keep=False)].head()


**Análisis:**  
Determinar si los registros repetidos son verdaderos duplicados o si tienen sentido dentro del contexto.


## 3.4 Validez y consistencia

Revisar valores mínimos, máximos, rangos imposibles, formatos inconsistentes y valores que no tengan sentido meteorológico según el diccionario de datos.


In [ ]:
# Ejemplo de apoyo para revisar rangos
data.describe().T


**Problemas encontrados:**  
- ...
- ...

**Decisiones propuestas:**  
- ...
- ...


## 3.5 Distribución de variables numéricas


In [ ]:
# Seleccionar variables numéricas
numeric_cols = data.select_dtypes(include=np.number).columns

# Histogramas
data[numeric_cols].hist(figsize=(16, 12), bins=30)
plt.tight_layout()
plt.show()


**Análisis:**  
Describir distribuciones llamativas, asimetrías, concentraciones y posibles valores atípicos.


## 3.6 Valores atípicos

Usar boxplots u otras técnicas cuando sean pertinentes. No eliminar un valor atípico únicamente porque sea extremo: primero determinar si representa un error o una observación válida.


In [ ]:
# Ejemplo: boxplot para una variable
# sns.boxplot(x=data["NOMBRE_VARIABLE"])
# plt.show()


## 3.7 Correlaciones entre variables numéricas


In [ ]:
corr = data[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matriz de correlación")
plt.show()


**Análisis:**  
- ¿Qué variables parecen estar más relacionadas con la variable objetivo?
- ¿Existen variables muy correlacionadas entre sí?
- ¿Podría existir redundancia o multicolinealidad?


# 4. Preparación de datos

Las transformaciones deben salir de los problemas encontrados durante el EDA.

Ejemplos:
- corregir tipos,
- eliminar duplicados justificados,
- imputar valores faltantes,
- transformar variables,
- crear nuevas características,
- seleccionar variables,
- codificar variables categóricas,
- escalar cuando corresponda.

> Evitar modificar el conjunto completo antes de la separación entrenamiento/prueba si esa transformación aprende parámetros de los datos. Para esos casos se deben usar **pipelines**.


## 4.1 Definición de la variable objetivo

Reemplazar `TARGET` por el nombre exacto de la columna que representa la temperatura máxima.


In [ ]:
TARGET = "REEMPLAZAR_CON_VARIABLE_OBJETIVO"

# X = variables predictoras
# y = variable objetivo
# X = data.drop(columns=[TARGET])
# y = data[TARGET]


## 4.2 Separación entrenamiento / prueba

El laboratorio exige:
- `test_size = 0.25`
- `random_state = 42`


In [ ]:
# Descomentar después de definir TARGET

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.25,
#     random_state=42
# )

# print("X_train:", X_train.shape)
# print("X_test:", X_test.shape)


# 5. Modelo 1 — Regresión lineal base

Construir una primera estrategia de preparación y justificarla.

### Hipótesis / estrategia del modelo 1
Escribir aquí:
- qué variables se usarán,
- qué tratamiento de faltantes se aplicará,
- si se escalarán variables,
- qué transformaciones se incluirán,
- por qué esta estrategia constituye una buena línea base.


In [ ]:
# EJEMPLO DE ESTRUCTURA DEL PIPELINE
# Ajustar las listas una vez realizado el EDA.

# numeric_features_m1 = [...]
# categorical_features_m1 = [...]

# numeric_transformer_m1 = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler())
# ])

# categorical_transformer_m1 = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("onehot", OneHotEncoder(handle_unknown="ignore"))
# ])

# preprocessor_m1 = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer_m1, numeric_features_m1),
#         ("cat", categorical_transformer_m1, categorical_features_m1)
#     ]
# )

# model_1 = Pipeline(steps=[
#     ("preprocessor", preprocessor_m1),
#     ("regressor", LinearRegression())
# ])


In [ ]:
# Entrenar modelo 1
# model_1.fit(X_train, y_train)

# pred_1 = model_1.predict(X_test)


## 5.1 Evaluación del modelo 1


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

# metrics_1 = regression_metrics(y_test, pred_1)
# metrics_1


**Interpretación del modelo 1:**  
Explicar qué significan los valores obtenidos y cuáles parecen ser las principales limitaciones del modelo.


# 6. Modelo 2 — Regresión lineal con una preparación diferente

El laboratorio exige al menos dos modelos de regresión lineal con **estrategias de preparación/ingeniería de características diferentes y justificadas**.

### Hipótesis / estrategia del modelo 2

Por ejemplo, aquí podrían probar:
- otra selección de variables,
- nuevas características,
- tratamiento distinto de atípicos,
- transformación de variables,
- eliminación de variables redundantes,
- otra estrategia de imputación.

La decisión concreta debe salir del EDA.


In [ ]:
# Construir pipeline del modelo 2

# model_2 = Pipeline(steps=[
#     ...
# ])


In [ ]:
# Entrenar y evaluar modelo 2

# model_2.fit(X_train, y_train)
# pred_2 = model_2.predict(X_test)
# metrics_2 = regression_metrics(y_test, pred_2)
# metrics_2


# 7. Comparación de modelos

Construir la tabla final con los dos mejores modelos analizados.


In [ ]:
# Ejemplo:
# comparacion = pd.DataFrame([
#     {"Modelo": "Modelo 1", **metrics_1},
#     {"Modelo": "Modelo 2", **metrics_2}
# ])

# comparacion.sort_values("RMSE")


## Selección del mejor modelo

**Modelo seleccionado:** ...

**Justificación:**  
Comparar principalmente RMSE, pero interpretar también MAE y R². Explicar por qué se escoge el modelo final y si la diferencia respecto al otro modelo es relevante.


# 8. Evaluación cualitativa e interpretación

## 8.1 Coeficientes del mejor modelo

Obtener los nombres de las características después del pipeline y asociarlos con los coeficientes de la regresión.


In [ ]:
# BASE PARA EXTRAER COEFICIENTES
# Ajustar al nombre del mejor pipeline.

# best_model = model_2
# preprocessor = best_model.named_steps["preprocessor"]
# regressor = best_model.named_steps["regressor"]

# feature_names = preprocessor.get_feature_names_out()

# coeficientes = (
#     pd.DataFrame({
#         "variable": feature_names,
#         "coeficiente": regressor.coef_
#     })
#     .assign(importancia_abs=lambda df: df["coeficiente"].abs())
#     .sort_values("importancia_abs", ascending=False)
# )

# coeficientes


### Interpretación

Responder:
- ¿Cuáles variables tienen los coeficientes de mayor magnitud?
- ¿Qué significa el signo positivo o negativo?
- ¿Qué interpretación puede darse en el contexto meteorológico?
- ¿La escala de las variables permite comparar directamente los coeficientes?


# 9. Verificación de supuestos de regresión lineal

Antes de hacer interpretaciones fuertes de los coeficientes, revisar los supuestos vistos en clase.

Incluir al menos los análisis correspondientes indicados en la práctica de verificación de supuestos.

### Posibles apartados
- linealidad,
- comportamiento de residuos,
- homocedasticidad,
- normalidad de residuos,
- independencia,
- multicolinealidad.

Agregar gráficos/pruebas y una conclusión para cada supuesto.


In [ ]:
# Residuos del mejor modelo
# best_pred = best_model.predict(X_test)
# residuals = y_test - best_pred

# Ejemplo: residuos vs predicción
# plt.scatter(best_pred, residuals, alpha=0.6)
# plt.axhline(0, linestyle="--")
# plt.xlabel("Valores predichos")
# plt.ylabel("Residuos")
# plt.title("Residuos vs valores predichos")
# plt.show()


# 10. Análisis de resultados

Responder explícitamente las preguntas solicitadas en el laboratorio.

## 10.1 ¿Cuál fue el valor de los diferentes coeficientes obtenidos en el mejor modelo?

**Respuesta:**  
...

## 10.2 ¿Cuál modelo ofrece el mejor rendimiento sobre test? ¿Cómo se interpretan RMSE, MAE y R²?

**Respuesta:**  
...

## 10.3 ¿Cuáles variables fueron seleccionadas? ¿Qué interpretación tienen en el problema?

**Respuesta:**  
...

## 10.4 Representación matemática de la regresión lineal

Escribir el modelo de forma general:

\[
\hat{y}=\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p
\]

Luego reemplazarlo por la ecuación correspondiente al modelo obtenido.

**Método utilizado para estimar los parámetros:**  
...


# 11. Sesgos posibles

Describir **dos tipos de sesgo** que podrían afectar este proyecto y explicar específicamente por qué podrían aparecer con estos datos.

### Sesgo 1: ...
**Explicación:** ...

### Sesgo 2: ...
**Explicación:** ...


# 12. Predicciones sobre el conjunto final sin etiqueta

Una vez seleccionado y validado el mejor modelo:
1. cargar `Datos Test Lab 1.csv`,
2. aplicar el mismo pipeline,
3. generar la predicción,
4. agregar la columna solicitada,
5. exportar el CSV conservando las demás columnas.


In [ ]:
final_test = pd.read_csv(FINAL_TEST_PATH)

print(final_test.shape)
final_test.head()


In [ ]:
# Generar predicciones finales
# final_predictions = best_model.predict(final_test)

# IMPORTANTE:
# reemplazar el nombre de la columna por el nombre EXACTO solicitado por el laboratorio.
# final_output = final_test.copy()
# final_output["NOMBRE_COLUMNA_PREDICCION"] = final_predictions

# OUTPUT_PATH = RESULTS_DIR / "Datos Test Lab 1_predicciones.csv"
# final_output.to_csv(OUTPUT_PATH, index=False)

# print("Archivo guardado en:", OUTPUT_PATH)


# 13. Conclusiones

Redactar de forma corta y concreta:

1. qué tan bien funcionó el modelo,
2. qué estrategia mejoró más los resultados,
3. cuáles variables fueron más relevantes,
4. qué limitaciones tiene el análisis,
5. cómo podría usarse este conocimiento en el contexto de AlpesPlanck.


# 14. Uso de herramientas de IA generativa

Esta sección es obligatoria si se utilizó IA.

## Declaración del uso

**Herramienta:** ...  
**Tipo de uso:** ayuda conceptual / generación inicial de código / depuración / redacción / etc.

## Prompts principales utilizados

1. `...`
2. `...`
3. `...`

## Análisis crítico del resultado

Responder al menos dos:
- ¿Qué partes generadas fueron correctas y útiles?
- ¿Qué errores, imprecisiones o limitaciones se identificaron?
- ¿Qué decisiones técnicas fueron modificadas y por qué?
- ¿Qué conceptos del curso permitieron evaluar o mejorar la respuesta?

**Respuesta:**  
...

## Aportes propios

Explicar qué fue desarrollado, modificado y decidido por los integrantes del grupo.

**Respuesta:**  
...


# 15. Checklist antes de entregar

- [ ] El notebook ejecuta de principio a fin sin errores.
- [ ] Todas las celdas tienen sus resultados visibles.
- [ ] El EDA contiene conclusiones y no solo gráficos.
- [ ] Las decisiones de preparación están justificadas.
- [ ] Se construyeron al menos dos regresiones lineales con estrategias diferentes.
- [ ] Se usaron pipelines.
- [ ] La separación usa `test_size=0.25` y `random_state=42`.
- [ ] Se comparan RMSE, MAE y R².
- [ ] Se presenta una tabla de coeficientes/importancia.
- [ ] Se verifican los supuestos de regresión lineal.
- [ ] Se responden todas las preguntas de análisis.
- [ ] Se discuten dos posibles sesgos.
- [ ] Se generan las predicciones finales.
- [ ] Se exporta el CSV requerido.
- [ ] Se documenta el uso de IA.
- [ ] Se exporta también el notebook a `.html`.
